<a href="https://www.kaggle.com/code/aayushparajuli03/twitter-roberta-experiments-ipynb?scriptVersionId=316704939" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Twitter-RoBERTa Experiments for Sentiment Classification

## Purpose

This notebook explores the performance of a **domain-specific transformer model (Twitter-RoBERTa)** for binary sentiment classification.

Unlike general-purpose models (e.g., DistilBERT), Twitter-RoBERTa is pre-trained on **Twitter-specific text**, making it more suitable for handling:
- Slang and informal language  
- Hashtags and mentions  
- Emojis and noisy social media text  

The goal is to evaluate whether **domain-specific pretraining improves performance and data efficiency** in sentiment analysis tasks.

---

## What This Notebook Does

- Trains Twitter-RoBERTa on sentiment datasets of varying sizes:
  - Small (~100k samples)
  - Medium (~200k samples)
  - Large (~400k samples)

- Keeps experimental settings consistent with previous DistilBERT experiments:
  - Same train/test split
  - Same evaluation metrics
  - Comparable hyperparameters

- Evaluates performance using:
  - Accuracy  
  - F1-score  
  - ROC-AUC  
  - Training time  

---

## Key Research Focus

This experiment is designed to answer:

- Does a **domain-specific transformer outperform a general-purpose model**?
- Does Twitter-RoBERTa achieve better results with **less training data**?
- How does it compare to DistilBERT in terms of:
  - Performance scaling  
  - Efficiency  
  - Stability  

---

## Role in Overall Project

This notebook is a key extension of the baseline study and will be used to:

- Compare against **DistilBERT dataset scaling results**
- Analyze **data efficiency across model architectures**
- Evaluate the impact of **domain-specific pretraining**
- Support further extension into **brand sentiment analysis and domain adaptation**

---

## Expected Contribution

This experiment aims to provide insights into:

> Whether domain-specific transformer models reduce the need for large datasets while improving sentiment classification performance on social-media-style text.

In [1]:
!uv pip install transformers datasets tqdm accelerate

Using Python 3.12.12 environment at: /usr
Audited 4 packages in 373ms


In [2]:
# !pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -U transformers accelerate

Looking in indexes: https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 94.0 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transf

Checking GPU

In [3]:
import torch
torch.cuda.is_available()

True

In [4]:
# !nvidia-smi

In [5]:
# ! pip install transformers[torch] datasets tqdm accelerate --only-binary :all: -i https://pypi.tuna.tsinghua.edu.cn/simple

# Dependencies

In [6]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, pipeline, BertTokenizerFast
from transformers import (
    DistilBertForSequenceClassification, 
    DistilBertTokenizerFast,
    AutoTokenizer,
    AutoModelForSequenceClassification
)
import time
import os
import json, pickle as pkl



# Loading Dataset

In [7]:
# ! pip install kaggle

get kaggle.json from the kaggle and add to working environment

In [8]:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

Importing twitter sentiment dataset

In [9]:
# !kaggle datasets download -d kazanova/sentiment140

if data set is downloaded using: !kaggle datasets download -d kazanova/sentiment140

In [10]:
# # extracting the compressed dataset

# from zipfile import ZipFile
# dataset = '/content/sentiment140.zip'

# with ZipFile(dataset, 'r') as zip:
#   zip.extractall()
#   print('The dataset is extracted successfully')

Use datasets/kazanova/sentiment140 from kaggle

In [11]:
# colab
# df = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1', header = None)

#colab + dataset saved on drive
# df = pd.read_csv('/content/drive/MyDrive/Sentiment Analysis Project/datasets/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1', header = None)

#kaggle
df = pd.read_csv(
    '/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv',
    encoding='ISO-8859-1',
    header=None
)

df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']

# Converting labels
df['target'] = df['target'].replace(4,1)

# Keeping only needed columns
df = df[['text', 'target']]


## Create 3 datasets

- Dataset 1: Small (50k per class → 100k total)
- Dataset 2: Medium (100k per class → 200k total) MAIN
- Dataset 3: Large (200k per class → 400k total)

In [12]:
df_small = df.groupby('target').sample(50000, random_state=42).reset_index(drop=True)

In [13]:
df_medium = df.groupby('target').sample(100000, random_state=42).reset_index(drop=True)

In [14]:
df_large = df.groupby('target').sample(200000, random_state=42).reset_index(drop=True)

Sanity check

In [15]:
print("Small:\n", df_small['target'].value_counts())
print("\nMedium:\n", df_medium['target'].value_counts())
print("\nLarge:\n", df_large['target'].value_counts())

Small:
 target
0    50000
1    50000
Name: count, dtype: int64

Medium:
 target
0    100000
1    100000
Name: count, dtype: int64

Large:
 target
0    200000
1    200000
Name: count, dtype: int64


In [16]:
# Uncomment below code to save the datasets

# df_small.to_csv("sentiment_small.csv", index=False)
# df_medium.to_csv("sentiment_medium.csv", index=False)
# df_large.to_csv("sentiment_large.csv", index=False)

# df_small.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_small.csv", index=False)
# df_medium.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_medium.csv", index=False)
# df_large.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_large.csv", index=False)

# REUSABLE Twitter-RoBERTa EXPERIMENT PIPELINE

## Metrics function

In [17]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)

  precision, recall,f1, _ = precision_recall_fscore_support(labels, preds, average = 'binary')
  acc = accuracy_score(labels, preds)

  probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
  roc = roc_auc_score(labels, probs)

  return {
      "accuracy": acc,
      "f1": f1,
      "roc_auc": roc,
      "precision": precision,
      "recall": recall
  }




MAIN reusable function

kaggle

Code suitable to plot:
- loss curves
- error analysis (predictions saved)
- proper logging for research plots
- reproducible experiment setup

In [18]:
# def run_experiment(
#     df,
#     dataset_name="dataset",
#     save_dir="/kaggle/working/models",
#     model_name="cardiffnlp/twitter-roberta-base-sentiment"
# ):

#     print(f"Running experiment on {dataset_name} using {model_name}")

#     # Save directory logic
#     exp_path = os.path.join(save_dir, dataset_name)
#     os.makedirs(exp_path, exist_ok=True)

    
#     # Train / Validation Split
    
#     train_texts, val_texts, train_labels, val_labels = train_test_split(
#         df['text'].tolist(),
#         df['target'].tolist(),
#         test_size=0.2,
#         random_state=42,
#         stratify=df['target']
#     )

#     with open(os.path.join(exp_path, "data_split_info.json"), "w") as f:
#         json.dump({
#             "train_size": len(train_texts),
#             "val_size": len(val_texts)
#         }, f, indent=4)

    
#     # Tokenizer 
    
#     tokenizer = AutoTokenizer.from_pretrained(model_name)

#     train_encodings = tokenizer(
#         train_texts,
#         truncation=True,
#         padding=True,
#         max_length=128
#     )

#     val_encodings = tokenizer(
#         val_texts,
#         truncation=True,
#         padding=True,
#         max_length=128
#     )

    
#     # Dataset Class
    
#     class SentimentDataset(torch.utils.data.Dataset):
#         def __init__(self, encodings, labels):
#             self.encodings = encodings
#             self.labels = labels

#         def __getitem__(self, idx):
#             item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
#             item["labels"] = torch.tensor(self.labels[idx])
#             return item

#         def __len__(self):
#             return len(self.labels)

#     train_dataset = SentimentDataset(train_encodings, train_labels)
#     val_dataset = SentimentDataset(val_encodings, val_labels)

    
#     # Model 
    
#     model = AutoModelForSequenceClassification.from_pretrained(
#         model_name,
#         num_labels=2,
#         ignore_mismatched_sizes=True
#     )

    
#     # Training Arguments
    
#     training_args = TrainingArguments(
#         output_dir=os.path.join(exp_path, "checkpoints"),
#         num_train_epochs=2,
#         per_device_train_batch_size=32,
#         per_device_eval_batch_size=32,
    
#         gradient_accumulation_steps=4,
#         learning_rate=2e-5,
    
#         eval_strategy="no",
#         save_strategy="no",
#         load_best_model_at_end=False,
    
#         fp16=True,
#         optim="adamw_torch",
#         dataloader_num_workers=2,
#         dataloader_pin_memory=True,
#         logging_steps=200,
#         report_to="none"
#     )

    
#     # Trainer
    
#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_dataset,
#         eval_dataset=val_dataset,
#         compute_metrics=compute_metrics
#     )

    
#     # Training
    
#     start = time.time()
#     trainer.train()
#     end = time.time()

#     trainer.save_state()

    
#     # Evaluation
    
#     results = trainer.evaluate()

#     final_results = {
#         "dataset": dataset_name,
#         "model": model_name,
#         "accuracy": results.get("eval_accuracy"),
#         "f1": results.get("eval_f1"),
#         "roc_auc": results.get("eval_roc_auc"),
#         "training_time_sec": end - start
#     }

    
#     # Save Outputs
    
#     trainer.save_model(exp_path)
#     tokenizer.save_pretrained(exp_path)

#     with open(os.path.join(exp_path, "results.json"), "w") as f:
#         json.dump(final_results, f, indent=4)

#     with open(os.path.join(exp_path, "results.pkl"), "wb") as f:
#         pkl.dump(final_results, f)

#     with open(os.path.join(exp_path, "training_args.json"), "w") as f:
#         json.dump(training_args.to_dict(), f, indent=4)

#     label_map = {0: "negative", 1: "positive"}
#     with open(os.path.join(exp_path, "label_map.json"), "w") as f:
#         json.dump(label_map, f, indent=4)

#     print(f"\nExperiment saved at: {exp_path}")

#     print("\n--- Files Generated ---")
#     for root, dirs, files in os.walk(exp_path):
#         for file in files:
#             print(os.path.join(root, file))

#     return final_results

In [19]:
def run_experiment(
    df,
    dataset_name="dataset",
    save_dir="/kaggle/working/models",
    model_name="cardiffnlp/twitter-roberta-base-sentiment"
):

    print(f"Running experiment on {dataset_name} using {model_name}")

    # Save directory
    exp_path = os.path.join(save_dir, dataset_name)
    os.makedirs(exp_path, exist_ok=True)

    # Train / Validation Split
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['text'].tolist(),
        df['target'].tolist(),
        test_size=0.2,
        random_state=42,
        stratify=df['target']
    )

    with open(os.path.join(exp_path, "data_split_info.json"), "w") as f:
        json.dump({
            "train_size": len(train_texts),
            "val_size": len(val_texts)
        }, f, indent=4)

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_encodings = tokenizer(
        train_texts,
        truncation=True,
        padding=True,
        max_length=128
    )

    val_encodings = tokenizer(
        val_texts,
        truncation=True,
        padding=True,
        max_length=128
    )

    # Dataset Class
    class SentimentDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels

        def __getitem__(self, idx):
            item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
            item["labels"] = torch.tensor(self.labels[idx])
            return item

        def __len__(self):
            return len(self.labels)

    train_dataset = SentimentDataset(train_encodings, train_labels)
    val_dataset = SentimentDataset(val_encodings, val_labels)

    # Model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        ignore_mismatched_sizes=True
    )

    # Training Arguments (UPDATED)
    training_args = TrainingArguments(
        output_dir=os.path.join(exp_path, "checkpoints"),

        num_train_epochs=2,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,

        gradient_accumulation_steps=4,
        learning_rate=2e-5,

        eval_strategy="epoch",
        save_strategy="no",

        fp16=True,
        optim="adamw_torch",

        dataloader_num_workers=2,
        dataloader_pin_memory=True,

        logging_strategy="steps",
        logging_steps=100,

        report_to="none"
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # Training
    start = time.time()
    trainer.train()
    end = time.time()

    trainer.save_state()

    # Evaluation
    results = trainer.evaluate()

    # Predictions (NEW - IMPORTANT FOR ERROR ANALYSIS)
    preds = trainer.predict(val_dataset)

    y_true = preds.label_ids
    y_pred = preds.predictions.argmax(axis=1)

    import pandas as pd

    df_preds = pd.DataFrame({
        "text": val_texts,
        "true": y_true,
        "pred": y_pred
    })

    df_preds.to_csv(os.path.join(exp_path, "predictions.csv"), index=False)

    # Save Loss Logs (NEW - FOR LOSS CURVES)
    log_history = trainer.state.log_history

    steps = []
    losses = []

    for log in log_history:
        if "loss" in log:
            steps.append(log["step"])
            losses.append(log["loss"])

    with open(os.path.join(exp_path, "loss_logs.json"), "w") as f:
        json.dump({"steps": steps, "loss": losses}, f, indent=4)

    # Final Results
    final_results = {
        "dataset": dataset_name,
        "model": model_name,
        "accuracy": results.get("eval_accuracy"),
        "f1": results.get("eval_f1"),
        "roc_auc": results.get("eval_roc_auc"),
        "training_time_sec": end - start
    }

    # Save Outputs
    trainer.save_model(exp_path)
    tokenizer.save_pretrained(exp_path)

    with open(os.path.join(exp_path, "results.json"), "w") as f:
        json.dump(final_results, f, indent=4)

    with open(os.path.join(exp_path, "results.pkl"), "wb") as f:
        pkl.dump(final_results, f)

    with open(os.path.join(exp_path, "training_args.json"), "w") as f:
        json.dump(training_args.to_dict(), f, indent=4)

    label_map = {0: "negative", 1: "positive"}
    with open(os.path.join(exp_path, "label_map.json"), "w") as f:
        json.dump(label_map, f, indent=4)

    print(f"\nExperiment saved at: {exp_path}")

    print("\n--- Files Generated ---")
    for root, dirs, files in os.walk(exp_path):
        for file in files:
            print(os.path.join(root, file))

    return final_results

# Run all experiments

In [20]:
import transformers
print(transformers.__version__)

5.7.0


In [21]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

CUDA available: True
GPU name: Tesla T4


In [22]:
# !nvidia-smi

In [23]:
TRAIN_SMALL = True
TRAIN_MEDIUM = False
TRAIN_LARGE = False



In [24]:
if TRAIN_MEDIUM:
    results_medium = run_experiment(
        df_medium,
        dataset_name="df_medium",
        model_name="cardiffnlp/twitter-roberta-base-sentiment"
    )
    result_medium

In [25]:
if TRAIN_LARGE:
    # results_large = run_experiment(df_large, "large_200k")
    results_large = run_experiment(
        df_large,
        dataset_name="df_large",
        model_name="cardiffnlp/twitter-roberta-base-sentiment"
    )
    results_large

In [26]:
if TRAIN_SMALL:
    results_small = run_experiment(
        df_small,
        dataset_name="df_small",
        model_name="cardiffnlp/twitter-roberta-base-sentiment"
    )
    results_small

Running experiment on df_small using cardiffnlp/twitter-roberta-base-sentiment


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1,Roc Auc,Precision,Recall
2.137043,0.602292,2,0.876050,0.876143,0.948333,0.875487,0.876800


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment saved at: /kaggle/working/models/df_small

--- Files Generated ---
/kaggle/working/models/df_small/training_args.json
/kaggle/working/models/df_small/loss_logs.json
/kaggle/working/models/df_small/results.json
/kaggle/working/models/df_small/tokenizer_config.json
/kaggle/working/models/df_small/training_args.bin
/kaggle/working/models/df_small/model.safetensors
/kaggle/working/models/df_small/predictions.csv
/kaggle/working/models/df_small/data_split_info.json
/kaggle/working/models/df_small/config.json
/kaggle/working/models/df_small/tokenizer.json
/kaggle/working/models/df_small/results.pkl
/kaggle/working/models/df_small/label_map.json
/kaggle/working/models/df_small/checkpoints/trainer_state.json


In [27]:
if TRAIN_SMALL:
    results_small = run_experiment(
    df_small,
    dataset_name="df_small",
    model_name="distilbert-base-uncased"
)

Running experiment on df_small using distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1,Roc Auc,Precision,Recall
2.861949,0.757192,2,0.834850,0.834212,0.912709,0.837448,0.831000


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment saved at: /kaggle/working/models/df_small

--- Files Generated ---
/kaggle/working/models/df_small/training_args.json
/kaggle/working/models/df_small/loss_logs.json
/kaggle/working/models/df_small/results.json
/kaggle/working/models/df_small/tokenizer_config.json
/kaggle/working/models/df_small/training_args.bin
/kaggle/working/models/df_small/model.safetensors
/kaggle/working/models/df_small/predictions.csv
/kaggle/working/models/df_small/data_split_info.json
/kaggle/working/models/df_small/config.json
/kaggle/working/models/df_small/tokenizer.json
/kaggle/working/models/df_small/results.pkl
/kaggle/working/models/df_small/label_map.json
/kaggle/working/models/df_small/checkpoints/trainer_state.json


# Compare results

In [28]:
# results_df = pd.DataFrame([
#     results_small,
#     results_medium,
#     results_large
# ])

# results_df

# Plots

Load Results Automatically

In [29]:
# import os
# import json
# import matplotlib.pyplot as plt

# # DistilBERT paths (your existing)
# DISTIL_PATHS = {
#     "small": "/kaggle/input/datasets/aayushparajuli03/small-result",
#     "medium": "/kaggle/input/notebooks/aayushparajuli03/sentiment-analysis/models/df_medium/results.json",
#     "large": "/kaggle/input/notebooks/aayushparajuli03/sentiment-analysis/models/large_200k/results.json",
# }

# # Twitter-RoBERTa paths
# ROBERTA_PATHS = {
#     "small": "/kaggle/input/notebooks/aayushparajuli03/twitter-roberta-experiments-ipynb/models/df_small/results.json",
#     "medium": "/kaggle/input/notebooks/aayushparajuli03/twitter-roberta-experiments-ipynb/models/df_medium/results.json",
#     "large": "/kaggle/input/notebooks/aayushparajuli03/twitter-roberta-experiments-ipynb/models/df_large/results.json",
# }

In [30]:
# def load_result(path):
#     if os.path.isdir(path):
#         path = os.path.join(path, "results.json")

#     with open(path, "r") as f:
#         return json.load(f)

# def load_all(paths):
#     results = {}
#     for name, path in paths.items():
#         results[name] = load_result(path)
#     return results

Convert to Structured Format

In [31]:
# distil = load_all(DISTIL_PATHS)
# roberta = load_all(ROBERTA_PATHS)

# datasets = ["small", "medium", "large"]

# def extract(metric, model_data):
#     return [model_data[d][metric] for d in datasets]

# distil_acc = extract("accuracy", distil)
# roberta_acc = extract("accuracy", roberta)

# distil_f1 = extract("f1", distil)
# roberta_f1 = extract("f1", roberta)

# distil_auc = extract("roc_auc", distil)
# roberta_auc = extract("roc_auc", roberta)

# distil_time = extract("training_time_sec", distil)
# roberta_time = extract("training_time_sec", roberta)

## Core Comparison Plots

Accuracy Comparison

In [32]:
# plt.figure()
# plt.plot(datasets, distil_acc, marker='o', label="DistilBERT")
# plt.plot(datasets, roberta_acc, marker='o', label="Twitter-RoBERTa")

# for i in range(len(datasets)):
#     plt.text(i, distil_acc[i], f"{distil_acc[i]:.3f}", ha='center')
#     plt.text(i, roberta_acc[i], f"{roberta_acc[i]:.3f}", ha='center')

# plt.xlabel("Dataset Size")
# plt.ylabel("Accuracy")
# plt.title("Accuracy vs Dataset Size")
# plt.legend()
# plt.grid()
# plt.show()

### Accuracy vs Dataset Size

The accuracy trends reveal a clear **domain advantage** for the specialized model.

- **Baseline Superiority:** Twitter-RoBERTa consistently maintains a higher performance floor. Notably, its accuracy on the *small dataset (87.7%)* surpasses DistilBERT’s performance even at the *large scale (85.2%)*.
  
- **Scaling Behavior:** Both models benefit from increased data; however, Twitter-RoBERTa approaches saturation earlier. The improvement from medium to large scale is marginal (*88.7% → 89.1%*), indicating diminishing returns.

**Insight:**  
> Domain-specific pretraining enables strong performance even under limited data regimes, reducing reliance on large-scale datasets.


F1 Comparison

In [33]:
# plt.figure()
# plt.plot(datasets, distil_f1, marker='o', label="DistilBERT")
# plt.plot(datasets, roberta_f1, marker='o', label="Twitter-RoBERTa")

# plt.xlabel("Dataset Size")
# plt.ylabel("F1 Score")
# plt.title("F1 Score vs Dataset Size")
# plt.legend()
# plt.grid()
# plt.show()

### F1-Score vs Dataset Size

The F1-score trends closely mirror accuracy, confirming robustness in predictions.

- **Metric Consistency:** The alignment between F1-score and accuracy indicates that performance is not skewed by class imbalance, but reflects genuine predictive capability.

- **Stable Performance Gap:** Twitter-RoBERTa maintains a consistent advantage of approximately **4–5 percentage points** across all dataset sizes.

**Insight:**  
> The performance gains are systematic and balanced across classes, reinforcing the effectiveness of domain-specific representations.


ROC-AUC Comparison

In [34]:
# plt.figure()
# plt.plot(datasets, distil_auc, marker='o', label="DistilBERT")
# plt.plot(datasets, roberta_auc, marker='o', label="Twitter-RoBERTa")

# plt.xlabel("Dataset Size")
# plt.ylabel("ROC-AUC")
# plt.title("ROC-AUC vs Dataset Size")
# plt.legend()
# plt.grid()
# plt.show()

### ROC-AUC vs Dataset Size

ROC-AUC evaluates the models’ ability to discriminate between sentiment classes.

- **High Discriminative Power:** Twitter-RoBERTa achieves a near-optimal ROC-AUC of **~0.96** on the large dataset.

- **Separation Capability:** The persistent gap between the models indicates that Twitter-RoBERTa forms more distinct decision boundaries.

**Insight:**  
> Domain-specific pretraining enhances the model’s ability to capture subtle sentiment cues, leading to superior class separability.


Training Time Comparison

In [35]:
# plt.figure()
# plt.plot(datasets, distil_time, marker='o', label="DistilBERT")
# plt.plot(datasets, roberta_time, marker='o', label="Twitter-RoBERTa")

# plt.xlabel("Dataset Size")
# plt.ylabel("Training Time (sec)")
# plt.title("Training Time vs Dataset Size")
# plt.legend()
# plt.grid()
# plt.show()

### Training Time Analysis

While performance improves with Twitter-RoBERTa, computational cost increases significantly.

- **Higher Computational Cost:** Training time scales sharply, reaching over **7,600 seconds** for the large dataset—approximately **2× slower** than DistilBERT.

- **Efficiency of DistilBERT:** DistilBERT demonstrates a more favorable scaling curve, making it suitable for time- and resource-constrained environments.

**Insight:**  
> There exists a clear trade-off between performance and computational efficiency.


### Why Does Twitter-RoBERTa Take Longer to Train?

The increased training time of Twitter-RoBERTa is primarily due to its **larger and more complex architecture**, rather than its domain-specific nature.

- **Model Size:** Twitter-RoBERTa (\~125M parameters, 12 layers) is significantly larger than DistilBERT (~66M parameters, 6 layers), resulting in more computations per training step.

- **No Distillation:** DistilBERT is a compressed model optimized for speed, while Twitter-RoBERTa is a full-capacity model designed for higher accuracy.

- **Richer Representations:** Twitter-RoBERTa captures complex features such as slang, emojis, and informal text, requiring deeper computation during training.

- **Optimization Complexity:** Larger models explore a broader parameter space, leading to slower and noisier convergence.

**Key Insight:**  
> The higher computational cost of Twitter-RoBERTa reflects a trade-off between efficiency and performance—achieving better accuracy at the expense of increased training time.

Time vs Accuracy (Efficiency Curve)

In [36]:
# plt.figure()
# plt.plot(distil_time, distil_acc, marker='o', label="DistilBERT")
# plt.plot(roberta_time, roberta_acc, marker='o', label="Twitter-RoBERTa")

# plt.xlabel("Training Time (sec)")
# plt.ylabel("Accuracy")
# plt.title("Training Time vs Accuracy")
# plt.legend()
# plt.grid()
# plt.show()

### Training Time vs Accuracy (Return on Investment)

This analysis highlights the **cost-effectiveness of training**.

- **RoBERTa Plateau:** Beyond the medium dataset (~4,000 seconds), Twitter-RoBERTa exhibits diminishing returns. Doubling training time results in only a **~0.4% accuracy gain**.

- **Optimal Trade-off Point:** The medium dataset provides the best balance between computational cost and predictive performance.

**Insight:**  
> Increasing computational investment does not proportionally translate into performance gains, emphasizing the importance of optimal stopping points.


Performance Gap Plot

Shows:

- how much domain-specific model wins

In [37]:
# gap = [r - d for r, d in zip(roberta_acc, distil_acc)]

# plt.figure()
# plt.plot(datasets, gap, marker='o')

# for i, val in enumerate(gap):
#     plt.text(i, val, f"{val:.3f}", ha='center')

# plt.xlabel("Dataset Size")
# plt.ylabel("Accuracy Gap")
# plt.title("Performance Gap (RoBERTa - DistilBERT)")
# plt.grid()
# plt.show()

### Performance Gap (Twitter-RoBERTa vs DistilBERT)

This analysis quantifies the advantage of domain-specific modeling.

- **Peak Advantage:** The performance gap is largest at the medium dataset size (**~0.044**), where domain alignment has maximum impact.

- **Partial Convergence:** At larger scales, the gap slightly narrows (**~0.039**), indicating that increased data allows DistilBERT to partially compensate.

**Insight:**  
> While large datasets improve general models, they do not fully bridge the gap created by domain-specific knowledge.


## Summary of Visual Insights

Across all plots, several consistent patterns emerge:

- Domain-specific models outperform general models at all scales  
- Data scaling improves performance but exhibits diminishing returns  
- Domain alignment significantly enhances data efficiency  
- Computational cost grows faster than performance gains  
- Optimal performance is achieved at intermediate dataset sizes  

**Key Takeaway:**  
> The combination of domain-specific pretraining and moderate dataset scaling yields the most efficient and effective sentiment analysis system.

# General vs Domain-Specific Transformer Models for Sentiment Analysis  
## A Study of Data Efficiency, Scaling Behavior, and Domain Alignment

---

## Abstract
This study presents a structured investigation into the role of domain-specific pretraining in transformer-based sentiment analysis. Moving beyond conventional model benchmarking, we analyze how **domain alignment interacts with dataset scaling** to influence performance, efficiency, and learning dynamics.

Using DistilBERT as a general-purpose baseline and Twitter-RoBERTa as a domain-specific model, we conduct controlled experiments across three dataset sizes. Results show that domain-specific pretraining significantly improves performance and data efficiency, often surpassing general models even at smaller scales. However, these gains come with increased computational cost and earlier saturation.

The study highlights a key paradigm in modern NLP: **model performance is governed not only by data volume but by alignment between training distribution and target domain**.

---

## 1. Introduction

Transformer-based models have revolutionized Natural Language Processing, achieving state-of-the-art results across diverse tasks. While scaling data and model size has been a dominant strategy, recent evidence suggests that **data relevance and domain alignment** may play an equally critical role.

In sentiment analysis, especially on social media data, challenges such as:
- Informal language
- Slang and abbreviations
- Emoji-driven sentiment
- Contextual sarcasm  

make general-purpose models less effective.

This study addresses the following research questions:

- How does domain-specific pretraining affect sentiment classification performance?
- Does it reduce the dependency on large-scale datasets?
- How do general and domain-specific models behave under dataset scaling?
- What are the computational trade-offs involved?

---

## 2. Background and Baselines

### 2.1 Classical and Machine Learning Baseline

Prior to transformer-based modeling, traditional approaches were implemented using:

- TF-IDF + Logistic Regression  
- Extensive preprocessing (slang normalization, emoji handling)

📎 Implementation:  
https://github.com/aayush-12321/Sentiment-Analysis-Project/blob/main/Sentiment%20Analysis.ipynb  

These models provided:
- Fast training
- Reasonable baseline performance  
- Limited capability in capturing contextual semantics  

---

### 2.2 Transformer Baseline: DistilBERT

DistilBERT serves as a **general-purpose transformer baseline**.

- Model: distilbert-base-uncased  
- Pretraining: General corpora (Wikipedia, BooksCorpus)  

📎 Implementation:  
https://github.com/aayush-12321/Sentiment-Analysis-Project/blob/main/Sentiment_Analysis_BERT.ipynb  

Key characteristics:
- Computationally efficient  
- Strong generalization  
- Limited domain awareness for social media text  

---

### 2.3 Domain-Specific Model: Twitter-RoBERTa

To address domain limitations, we introduce:

- Model: cardiffnlp/twitter-roberta-base-sentiment  
- Pretraining: Large-scale Twitter corpus  

This model is inherently optimized for:
- Slang and abbreviations  
- Emoji semantics  
- Informal sentence structures  

---

## 3. Methodology

### 3.1 Task Definition
- Binary Sentiment Classification  
- Classes: {0: Negative, 1: Positive}  

---

### 3.2 Experimental Design

To isolate the effect of domain-specific pretraining:

- Identical datasets used across models  
- Same preprocessing pipeline  
- Same hyperparameters  
- Only model architecture differs  

#### Dataset Scaling:

| Dataset | Size |
|--------|------|
| df_small | ~100k |
| df_medium | ~200k |
| df_large | ~400k |

---

### 3.3 Training Configuration

| Parameter | Value |
|----------|------|
| Epochs | 2 |
| Effective Batch Size | 256 |
| Learning Rate | 2e-5 |
| Optimizer | AdamW |
| Precision | FP16 |
| Max Length | 128 |

---

### 3.4 Evaluation Metrics

- Accuracy  
- F1 Score  
- ROC-AUC  
- Training Time  

---

## 4. Results

### 4.1 Comparative Performance

| Dataset | Model | Accuracy | F1 Score | ROC-AUC | Training Time (sec) |
|--------|------|---------|---------|--------|--------------------|
| df_small | DistilBERT | 0.8377 | 0.8315 | 0.9203 | 1319 |
| df_small | Twitter-RoBERTa | **0.8770** | **0.8770** | **0.9487** | 1909 |
| df_medium | DistilBERT | 0.8426 | 0.8427 | 0.9220 | 1852 |
| df_medium | Twitter-RoBERTa | **0.8868** | **0.8862** | **0.9549** | 3808 |
| df_large | DistilBERT | 0.8515 | 0.8511 | 0.9294 | 3709 |
| df_large | Twitter-RoBERTa | **0.8905** | **0.8899** | **0.9578** | 7636 |

---

## 5. Analysis

### 5.1 Domain Alignment vs Data Scaling

A key observation is that **Twitter-RoBERTa at 100k outperforms DistilBERT at 400k**.

This suggests:
> Domain alignment can substitute for large-scale data.

Implication:
- Performance is not purely a function of data size
- **Relevant data distribution is more valuable than raw volume**

---

### 5.2 Data Efficiency

- DistilBERT shows gradual improvement with increasing data
- Twitter-RoBERTa achieves high performance early

This indicates:
- General models are **data-dependent**
- Domain-specific models are **knowledge-dependent**

---

### 5.3 Diminishing Returns

Both models exhibit saturation:

- Largest gains occur from 100k → 200k  
- Marginal improvements beyond 200k  

However:
- Saturation occurs earlier for Twitter-RoBERTa  

Interpretation:
> Domain knowledge reduces the need for additional data but limits further gains

---

### 5.4 Computational Trade-offs

| Model | Strength | Limitation |
|------|--------|-----------|
| DistilBERT | Efficient | Lower performance |
| Twitter-RoBERTa | High accuracy | Higher cost |

Training time increases:
- ~2x with each dataset doubling  
- Without proportional performance gain  

---

### 5.5 Optimization Behavior

- DistilBERT:
  - Smooth convergence  
  - Higher final loss  

- Twitter-RoBERTa:
  - Noisier training  
  - Lower final loss (~1.96)  
  - Better representation learning  

---

## 6. Discussion

### 6.1 Rethinking Scaling Laws

Traditional deep learning assumes:
> More data → Better performance

This study refines that view:

> Better-aligned data → Faster and stronger performance gains

---

### 6.2 Implicit Knowledge Transfer

Twitter-RoBERTa benefits from:
- Pre-learned sentiment cues  
- Contextual understanding of informal language  

Thus:
- Fine-tuning becomes **adaptation**, not learning from scratch  

---

### 6.3 Practical Insight

For real-world systems:

- Medium datasets (~200k) often provide optimal trade-off  
- Domain-specific models should be preferred when available  
- Scaling data beyond a point yields diminishing returns  

---

## 7. Key Contributions

- Demonstrates importance of domain alignment in NLP  
- Quantifies data efficiency of domain-specific transformers  
- Provides empirical evidence of diminishing returns  
- Establishes cost-performance trade-offs in scaling  

---

## 8. Limitations

- Limited training epochs  
- No hyperparameter tuning  
- No statistical testing  
- No domain adaptation (yet)  

---

## 9. Future Work

- Domain adaptation on brand-specific datasets  
- Error analysis (sarcasm, emoji, slang)  
- Extended training and tuning  
- Cross-domain evaluation  

---

## 10. Conclusion

This study demonstrates that:

- Domain-specific models significantly outperform general models  
- Data scaling improves performance but saturates  
- Domain alignment reduces dependency on large datasets  
- Computational cost grows faster than performance  

**Final Insight:**
> In modern NLP, the effectiveness of a model depends as much on *where it was trained* as on *how much data it sees*.

---

## 11. Reproducibility

- Framework: Hugging Face Transformers  
- Models: DistilBERT, Twitter-RoBERTa  
- Hardware: GPU (T4, FP16)  
- Includes:
  - Preprocessing  
  - Training pipeline  
  - Evaluation  
  - Logging  

In [38]:
# !git config --global user.name "aayush-12321"
# !git config --global user.email "aayushparajuli23@gmail.com"

In [39]:
!